# Fine-Tune VadCLIP Bằng Prompt (Colab)

Notebook chuyên cho **các hướng prompt finetune**. Mỗi hướng nằm ở **một cell train riêng**.

## Quy ước chung

- **From scratch**: H1/2a/2b/3/2a+3 — `--use-pretrained-model false`
- **Warm-start delta-only (B)**: §15/§16 — load 2b/2a, freeze VAD; học **soft prompt + `Δ`**
- **Warm-start continue (C)**: §18/§19 — load 2b/2a, học tiếp **VAD + soft prompt + `Δ`**
- Learnable soft prompt (`text_prompt_embeddings`) **không đóng băng** ở các hướng residual
- Đầu vào prompt bank: output của `analyze_vadclip_prompt_text_space.ipynb`

## Các hướng

| Hướng | Ý nghĩa |
|---|---|
| 1 | Top-1 representative prompt |
| 2a | Diversity Top-K **weighted multi-prompt** (giữ K kênh) |
| 2b | **Prototype `p_c`**: gộp Top-K embedding → 1 vector/class |
| 3 | Residual trên encode(prompt) **from scratch** (VAD + soft prompt + `Δ`) |
| 3-B | Warm-start 2b, freeze VAD; học soft prompt + `Δ` |
| 3-C | Warm-start 2b, continue VAD + soft prompt + `Δ` |
| 2a+3 | Multi-prompt residual **from scratch** |
| 2a+3-B | Warm-start 2a, freeze VAD; học soft prompt + `Δ` |
| 2a+3-C | Warm-start 2a, continue VAD + soft prompt + `Δ` |

Sau train: `analyze_vadclip_class_prompt_alignment.ipynb`.


## 1. Mount Drive Và Đường Dẫn


In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT
PRETRAINED_MODEL = PROJECT_ROOT / 'model_ucf.pth'

RESULTS_ROOT = PROJECT_ROOT / 'code' / 'results'
EVALUATE_RESULTS_DIR = RESULTS_ROOT / 'evaluate'
ANALYZE_RESULTS_DIR = RESULTS_ROOT / 'analyze'
for result_dir in [RESULTS_ROOT, EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR]:
    result_dir.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

print('Project root:', PROJECT_ROOT)
print('Source dir:', SRC_DIR)
print('Feature root:', FEATURE_ROOT)
print('Results root:', RESULTS_ROOT)
print('Evaluate results:', EVALUATE_RESULTS_DIR)
print('Analyze results:', ANALYZE_RESULTS_DIR)
print('CWD:', Path.cwd())
print('Rule: all prompt finetune runs train FROM SCRATCH (no checkpoint resume).')


## 2. Cài Dependencies


In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib
print('Dependencies ready')


## 3. Kiểm Tra File Bắt Buộc


In [ ]:
required_paths = [
    SRC_DIR / 'model.py',
    SRC_DIR / 'model_description.py',
    SRC_DIR / 'model_multi_prompt.py',
    SRC_DIR / 'model_temporal_proposal.py',
    SRC_DIR / 'ucf_train_multi_prompt.py',
    SRC_DIR / 'model_prompt_replacement.py',
    SRC_DIR / 'ucf_train_prompt_replacement.py',
    SRC_DIR / 'ucf_option_prompt_replacement.py',
    SRC_DIR / 'model_prompt_residual.py',
    SRC_DIR / 'ucf_train_prompt_residual.py',
    SRC_DIR / 'ucf_option_prompt_residual.py',
    SRC_DIR / 'model_multi_prompt_residual.py',
    SRC_DIR / 'ucf_train_multi_prompt_residual.py',
    SRC_DIR / 'ucf_option_multi_prompt_residual.py',
    SRC_DIR / 'ucf_option_multi_prompt.py',
    SRC_DIR / 'ucf_option_class_prototype.py',
    SRC_DIR / 'ucf_train_class_prototype.py',
    SRC_DIR / 'ucf_test_description.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'ucf_analyze_manual_prompts.py',
    SRC_DIR / 'utils' / 'prompt_replacement.py',
    SRC_DIR / 'utils' / 'topk_tracker.py',
    SRC_DIR / 'utils' / 'tools.py',
    SRC_DIR / 'utils' / 'layers.py',
    SRC_DIR / 'utils' / 'ucf_detectionMAP.py',
    SRC_DIR / 'clip' / 'clip.py',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_segment_ucf.npy',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_label_ucf.npy',
    PRETRAINED_MODEL,
    PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required files:\n' + '\n'.join(missing) +
        '\n\nNeu thieu ucf_weighted_class_prompts.json, chay analyze_vadclip_prompt_text_space.ipynb truoc.'
    )

available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if not DRIVE_FEATURE_ROOT.exists() and available_feature_archive is None:
    archive_names = ', '.join(path.name for path in DRIVE_FEATURE_ARCHIVES)
    raise FileNotFoundError(f'Missing UCFClipFeatures folder or one archive: {archive_names}')

print('All required files exist.')


## 4. Copy Feature Sang Runtime Local


In [ ]:
import shutil
import subprocess
import time

subprocess.run(['df', '-h', '/content'], check=False)
available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if available_feature_archive is not None:
    subprocess.run(['du', '-sh', str(available_feature_archive)], check=False)
else:
    subprocess.run(['du', '-sh', str(DRIVE_FEATURE_ROOT)], check=False)

start = time.time()
if available_feature_archive is not None:
    local_archive = Path('/content') / available_feature_archive.name
    if not local_archive.exists() or local_archive.stat().st_size != available_feature_archive.stat().st_size:
        print('Copying archive to local runtime:', available_feature_archive)
        shutil.copy2(available_feature_archive, local_archive)
    else:
        print('Local archive already exists with matching size:', local_archive)

    print('Extracting archive:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)

    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive must contain top-level folder UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run([
            'rsync', '-ah', '--info=progress2',
            f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'
        ], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Prepared local features in {(time.time() - start) / 60:.1f} minutes')
print('Feature root:', FEATURE_ROOT)


## 5. Kiểm Tra Độ Phủ Feature


In [ ]:
import csv
from collections import Counter

def check_feature_coverage(csv_path, feature_root, preview=20):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = []
    for row in rows:
        path = feature_root / row['path']
        if not path.exists():
            missing.append(row)

    print(f'{csv_path.name}: rows={len(rows)}, missing_files={len(missing)}')
    if missing:
        print('Missing by label:', dict(Counter(row['label'] for row in missing)))
        for row in missing[:preview]:
            print(f"  {row.get('video_id', '')},{row['label']},{row['path']}")
        raise FileNotFoundError(f'{csv_path.name} references missing feature files.')

for csv_path in [
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
]:
    check_feature_coverage(csv_path, FEATURE_ROOT)


## 6. Smoke Test Feature Load


In [ ]:
import csv
import numpy as np

train_list = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv'
rows = list(csv.DictReader(train_list.open(encoding='utf-8')))
sample = rows[0]
feat_path = FEATURE_ROOT / sample['path']
feat = np.load(feat_path)
print('Sample video:', sample.get('video_id', sample['path']))
print('Label:', sample['label'])
print('Feature path:', feat_path)
print('Feature shape:', feat.shape, '| dtype:', feat.dtype)


## 7. Cấu Hình Chung Cho 3 Hướng

Áp dụng cho mọi cell train bên dưới. **Luôn from-scratch** (`USE_PRETRAINED_MODEL = False`).


In [ ]:
import json
import re
import shlex
import subprocess
from copy import deepcopy
from pathlib import Path

FINETUNE_RESULTS = []  # list of metric dicts printed after each direction


def run_command(cmd, capture=False):
    """Stream combined stdout/stderr and preserve the child traceback on failure."""
    printable_cmd = ' '.join(shlex.quote(part) for part in cmd)
    print('Running:', printable_cmd, flush=True)
    process = subprocess.Popen(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        output_lines.append(line)
    return_code = process.wait()
    output = ''.join(output_lines)
    if return_code != 0:
        tail = ''.join(output_lines[-80:])
        raise RuntimeError(
            f'Child process failed with exit code {return_code}.\n'
            f'Last child-process output:\n{tail}'
        )
    return output if capture else ''


def _parse_block(text, block_name):
    pattern = rf'{re.escape(block_name)}\s*\n((?:\s{{2}}\S.*\n)+)'
    match = re.search(pattern, text)
    if not match:
        return {}
    metrics = {}
    for line in match.group(1).splitlines():
        line = line.strip()
        if ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        # e.g. "87.15 (classifier)" or "30.03"
        num = re.match(r'([+-]?\d+(?:\.\d+)?)', value)
        if num:
            metrics[key] = float(num.group(1))
            if key == 'best_auc' and '(' in value:
                metrics['best_branch'] = value.split('(', 1)[1].rstrip(')')
    return metrics


def parse_evaluate_stdout(stdout_text):
    """Parse ucf_evaluate.py console output into baseline/model metrics."""
    baseline = _parse_block(stdout_text, 'Baseline')
    # evaluate prints "Description-guided" for finetuned model
    model = _parse_block(stdout_text, 'Description-guided')
    if not model:
        model = _parse_block(stdout_text, 'Description')
    comparison = {}
    for line in stdout_text.splitlines():
        if line.startswith('Delta,'):
            cols = line.split(',')
            keys = ['best_auc', 'ano_auc', 'mAP@0.1', 'mAP@0.2', 'mAP@0.3', 'mAP@0.4', 'mAP@0.5', 'avg_mAP']
            for key, raw in zip(keys, cols[1:]):
                try:
                    comparison[key] = float(raw)
                except ValueError:
                    pass
    return {
        'baseline': baseline,
        'model': model,
        'delta_vs_baseline': comparison,
    }


def print_finetune_result(direction_name, variant_name, parsed, extra=None):
    """Pretty-print one finetune run result to console."""
    model = parsed.get('model') or {}
    baseline = parsed.get('baseline') or {}
    delta = parsed.get('delta_vs_baseline') or {}

    print('\n' + '#' * 100, flush=True)
    print(f'FINETUNE RESULT | {direction_name}', flush=True)
    print(f'variant: {variant_name}', flush=True)
    print('#' * 100, flush=True)

    def row(label, d, keys):
        parts = [f'{k}={d.get(k, float("nan")):.2f}' if k in d else f'{k}=NA' for k in keys]
        print(f'  {label:<18} ' + ' | '.join(parts), flush=True)

    keys_main = ['classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP']
    print('Baseline (checkpoint model_ucf.pth):', flush=True)
    row('baseline', baseline, keys_main)
    print('Finetuned model:', flush=True)
    row('finetuned', model, keys_main)
    if 'best_auc' in model:
        print(
            f"  best_auc={model.get('best_auc'):.2f} "
            f"({model.get('best_branch', 'n/a')})",
            flush=True,
        )
    if delta:
        print('Delta vs baseline (best_auc / ano_auc / avg_mAP):', flush=True)
        print(
            f"  best_auc={delta.get('best_auc', float('nan')):+.2f} | "
            f"ano_auc={delta.get('ano_auc', float('nan')):+.2f} | "
            f"avg_mAP={delta.get('avg_mAP', float('nan')):+.2f}",
            flush=True,
        )
    if extra:
        for key, value in extra.items():
            print(f'  {key}: {value}', flush=True)
    print('#' * 100 + '\n', flush=True)

    record = {
        'direction': direction_name,
        'variant_name': variant_name,
        'baseline': baseline,
        'model': model,
        'delta_vs_baseline': delta,
    }
    if extra:
        record['extra'] = extra
    FINETUNE_RESULTS.append(record)
    return record


def print_all_finetune_results():
    """Print a compact comparison table of all completed directions."""
    print('\n' + '=' * 100, flush=True)
    print('ALL FINETUNE RESULTS (console summary)', flush=True)
    print('=' * 100, flush=True)
    if not FINETUNE_RESULTS:
        print('No finetune results recorded yet.', flush=True)
        return
    header = f"{'direction':<28} {'cls_auc':>8} {'cls_ap':>8} {'aln_auc':>8} {'ano_auc':>8} {'avg_mAP':>8} {'d_best':>8}"
    print(header, flush=True)
    print('-' * len(header), flush=True)
    for item in FINETUNE_RESULTS:
        m = item.get('model') or {}
        d = item.get('delta_vs_baseline') or {}
        print(
            f"{item.get('direction',''):<28} "
            f"{m.get('classifier_auc', float('nan')):8.2f} "
            f"{m.get('classifier_ap', float('nan')):8.2f} "
            f"{m.get('alignment_auc', float('nan')):8.2f} "
            f"{m.get('ano_auc', float('nan')):8.2f} "
            f"{m.get('avg_mAP', float('nan')):8.2f} "
            f"{d.get('best_auc', float('nan')):+8.2f}",
            flush=True,
        )
    print('=' * 100 + '\n', flush=True)


# ---- Shared prompt bank (from analyze_vadclip_prompt_text_space.ipynb) ----
PROMPT_BANK_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'
SELECTED_TOP1_JSON = PROJECT_ROOT / 'code' / 'ucf_selected_representative_class_prompts.json'

PROMPT_MODE = 'manual_caption'
VARIANT_BASE = 'nounverb_v2_balanced_topk5'

# Hard rule: never resume / never load a finetuned checkpoint as init.
USE_PRETRAINED_MODEL = False
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'

MAX_EPOCH = 10
LR = 2e-5
SCHEDULER_MILESTONES = ['4', '8']
EARLY_STOPPING_PATIENCE = 3
LAMBDA_CONSISTENCY = 0.01

# Keep regularizers off for first isolation runs; enable later if needed.
LAMBDA_ABNORMAL_BG = 0.0
ABNORMAL_BG_MARGIN = 0.15
ABNORMAL_BG_WARMUP_EPOCHS = 1
LAMBDA_NORMAL_TOPK = 0.0
LAMBDA_RANKING_MARGIN = 0.0

# Hướng 3: L2 on ||Delta_c||^2
LAMBDA_RESIDUAL = 1e-3

if not PROMPT_BANK_JSON.exists():
    raise FileNotFoundError(
        f'Missing prompt bank: {PROMPT_BANK_JSON}\n'
        'Run analyze_vadclip_prompt_text_space.ipynb first.'
    )

with PROMPT_BANK_JSON.open('r', encoding='utf-8') as f:
    PROMPT_BANK = json.load(f)

META = PROMPT_BANK.get('metadata', {})
Path('model').mkdir(exist_ok=True)
RUN_OUTPUTS = {}

print('Prompt bank:', PROMPT_BANK_JSON)
print('Selected top-1 JSON (if exists):', SELECTED_TOP1_JSON.exists(), SELECTED_TOP1_JSON)
print('Selection method:', META.get('selection_method'))
print('TopK / diversity:', META.get('topk'), META.get('diversity_mode'), META.get('diversity_threshold'))
print('Prompts/class:', META.get('prompts_per_class'))
print('USE_PRETRAINED_MODEL:', USE_PRETRAINED_MODEL)
print('Prompt mode:', PROMPT_MODE)
print('LAMBDA_RESIDUAL:', LAMBDA_RESIDUAL)


def build_top1_prompt_json():
    # Prefer analyze Top-1 file; otherwise derive from weighted bank.
    if SELECTED_TOP1_JSON.exists():
        print('Using selected Top-1 bank:', SELECTED_TOP1_JSON)
        return SELECTED_TOP1_JSON

    out_path = PROJECT_ROOT / 'code' / f'ucf_top1_from_{PROMPT_BANK_JSON.stem}.json'
    payload = {
        'metadata': {
            **deepcopy(PROMPT_BANK.get('metadata', {})),
            'name': 'top1_from_weighted_bank',
            'source_prompt_bank': str(PROMPT_BANK_JSON),
            'training_mode': 'top1',
        },
        'classes': {},
    }
    print('Derived Top-1 prompts from weighted bank:')
    for class_name, prompts in PROMPT_BANK['classes'].items():
        weights = PROMPT_BANK['prompt_weights'][class_name]
        best_idx = max(range(len(weights)), key=lambda i: weights[i])
        payload['classes'][class_name] = [prompts[best_idx]]
        print(f'  {class_name:<13} w={weights[best_idx]:.4f} | {prompts[best_idx]}')
    out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    return out_path


def run_multi_prompt_experiment(direction_name, variant_suffix, prompt_json, prompt_weights_json=None):
    # Train + eval + diagnostics. Always from scratch.
    if USE_PRETRAINED_MODEL:
        raise RuntimeError('Refuse to train with USE_PRETRAINED_MODEL=True')

    reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
    variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_{reg_tag}'

    epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
    output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
    diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'
    manual_prompt_diag_dir = ANALYZE_RESULTS_DIR / f'ucf_manual_prompt_diagnostics_{variant_name}_scratch'
    topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}_scratch'

    # Remove stale checkpoints so this run cannot accidentally resume.
    for stale in [Path(checkpoint_path), Path(save_cur_path)]:
        if stale.exists():
            stale.unlink()
            print('Removed stale checkpoint:', stale)

    common_prompt_args = [
        '--prototype-source', 'manual',
        '--prototype-json', str(prompt_json),
        '--prompt-mode', PROMPT_MODE,
    ]
    if prompt_weights_json is not None:
        common_prompt_args.extend(['--prompt-weights-json', str(prompt_weights_json)])

    train_cmd = [
        'python', 'ucf_train_multi_prompt.py',
        '--feature-root', str(FEATURE_ROOT),
        *common_prompt_args,
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lambda-consistency', str(LAMBDA_CONSISTENCY),
        '--consistency-top-k-ratio', '0.15',
        '--regularize-score-branch', 'both',
        '--regularize-top-k-ratio', '0.15',
        '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
        '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
        '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
        '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
        '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
        '--ranking-margin', '0.20',
        '--lr', str(LR),
        '--max-epoch', str(MAX_EPOCH),
        '--scheduler-milestones', *SCHEDULER_MILESTONES,
        '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--track-topk', 'true',
        '--topk-tracking-output-dir', str(topk_tracking_dir),
        '--topk-tracking-branches', 'both',
        '--topk-tracking-scope', 'abnormal',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]

    print('\n' + '=' * 100)
    print(direction_name)
    print('variant:', variant_name)
    print('prompt_json:', prompt_json)
    print('prompt_weights_json:', prompt_weights_json)
    print('from_scratch: use-pretrained-model=false')
    run_command(train_cmd, capture=False)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'multi_prompt',
        *common_prompt_args,
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    eval_stdout = run_command(eval_cmd, capture=True)
    parsed = parse_evaluate_stdout(eval_stdout)
    result_record = print_finetune_result(direction_name, variant_name, parsed)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'multi_prompt',
        *common_prompt_args,
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    run_command(analyze_cmd, capture=False)

    manual_cmd = [
        'python', 'ucf_analyze_manual_prompts.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--manual-model-path', output_model,
        '--prompt-json', str(prompt_json),
        '--prompt-mode', PROMPT_MODE,
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--output-dir', str(manual_prompt_diag_dir),
    ]
    if prompt_weights_json is not None:
        manual_cmd.extend(['--prompt-weights-json', str(prompt_weights_json)])
    run_command(manual_cmd, capture=False)

    info = {
        'direction': direction_name,
        'variant_name': variant_name,
        'model': output_model,
        'epoch_dir': epoch_dir,
        'diagnostics_dir': str(diagnostics_dir),
        'manual_prompt_diag_dir': str(manual_prompt_diag_dir),
        'topk_tracking_dir': str(topk_tracking_dir),
        'prompt_json': str(prompt_json),
        'prompt_weights_json': None if prompt_weights_json is None else str(prompt_weights_json),
        'use_pretrained_model': False,
        'trained_from_scratch': True,
        'metrics': result_record,
    }
    RUN_OUTPUTS[direction_name] = info
    summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
    summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
    metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
    metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Saved summary:', summary_path)
    print('Saved metrics:', metrics_path)
    print_all_finetune_results()
    return info


## 8. Preview Prompt Bank


In [ ]:
print('Prompt bank preview (weight-sorted):')
for class_name, prompts in PROMPT_BANK['classes'].items():
    weights = PROMPT_BANK.get('prompt_weights', {}).get(class_name)
    if weights is None:
        weights = [1.0 / len(prompts)] * len(prompts)
    pairs = sorted(zip(weights, prompts), reverse=True)
    print(f'\n{class_name}:')
    for weight, prompt in pairs:
        print(f'  {weight:.4f} | {prompt}')


## 9. Hướng 1 — Top-1 Representative Prompt

**Mục tiêu:** Mỗi class chỉ dùng 1 prompt đại diện tốt nhất (sau text-space scoring).

- Train **from scratch**
- Không load checkpoint
- Prompt JSON: `ucf_selected_representative_class_prompts.json` (hoặc derive từ weighted bank)

**Kết quả lưu:**
- Evaluate: `code/results/evaluate/ucf_checkpoint_diagnostics_multi_prompt_*`
- Analyze: `code/results/analyze/ucf_manual_prompt_diagnostics_*` và `topk_tracking_*`
- Summary: `code/results/prompt_guided_*.json`


In [ ]:
# === HUONG 1: Top-1 Representative Prompt ===
# Logic giống notebook cũ (train -> evaluate -> analyze checkpoints -> manual prompt diag).
# Kết quả ghi vào code/results/evaluate và code/results/analyze.
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()

top1_json = build_top1_prompt_json()
direction_name = 'huong1_top1'
variant_suffix = 'huong1_top1'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_{reg_tag}'

prompt_json = top1_json
prompt_weights_json = None

epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'
manual_prompt_diag_dir = ANALYZE_RESULTS_DIR / f'ucf_manual_prompt_diagnostics_{variant_name}_scratch'
topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}_scratch'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir, manual_prompt_diag_dir, topk_tracking_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(prompt_json),
    '--prompt-mode', PROMPT_MODE,
]

train_cmd = [
    'python', 'ucf_train_multi_prompt.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', str(LAMBDA_CONSISTENCY),
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', '0.20',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('prompt_json:', prompt_json)
print('prompt_weights_json:', prompt_weights_json)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('Manual prompt diagnostics ->', manual_prompt_diag_dir)
print('Top-k tracking ->', topk_tracking_dir)
print('from_scratch: use-pretrained-model=false')
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(direction_name, variant_name, parsed)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

manual_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', output_model,
    '--prompt-json', str(prompt_json),
    '--prompt-mode', PROMPT_MODE,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(manual_prompt_diag_dir),
]
run_command(manual_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'manual_prompt_diag_dir': str(manual_prompt_diag_dir),
    'topk_tracking_dir': str(topk_tracking_dir),
    'prompt_json': str(prompt_json),
    'prompt_weights_json': None,
    'use_pretrained_model': False,
    'trained_from_scratch': True,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('\nAnalyze prompt training outputs:')
print('  mode: top1')
print('  model:', output_model)
print('  epoch checkpoints:', epoch_dir)
print('  checkpoint diagnostics:', diagnostics_dir)
print('  prompt diagnostics:', manual_prompt_diag_dir)
print('  top-k tracking:', topk_tracking_dir)
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 10. Hướng 2a — Diversity Top-K Weighted Multi-Prompt (giữ K kênh)

**Mục tiêu:** Dùng nhiều prompt/class đã chọn Top-K + diversity filter + weight.

Đây là multi-prompt: **giữ K embedding riêng**, aggregate bằng weight / logsumexp — **chưa** gộp thành một vector `p_c`.

- Train **from scratch**
- Không load checkpoint
- Prompt JSON + weights: `ucf_weighted_class_prompts.json`
- Learnable prefix/postfix vẫn tồn tại trong kiến trúc nhưng **được đóng băng** trong lần train này

**Kết quả lưu:**
- Evaluate: `code/results/evaluate/ucf_checkpoint_diagnostics_multi_prompt_*`
- Analyze: `code/results/analyze/ucf_manual_prompt_diagnostics_*` và `topk_tracking_*`
- Summary: `code/results/prompt_guided_*.json`


In [ ]:
# === HUONG 2a: Diversity Top-K + weighted multi-prompt ===
# Logic giống notebook cũ §22 (weighted): train -> evaluate -> analyze -> manual prompt diag.
# Kết quả ghi vào code/results/evaluate và code/results/analyze.
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()

direction_name = 'huong2_topk_weighted_frozen_prompt'
variant_suffix = 'huong2_topk_weighted_frozen_prompt'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_{reg_tag}'

prompt_json = PROMPT_BANK_JSON
prompt_weights_json = PROMPT_BANK_JSON

epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'
manual_prompt_diag_dir = ANALYZE_RESULTS_DIR / f'ucf_manual_prompt_diagnostics_{variant_name}_scratch'
topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}_scratch'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir, manual_prompt_diag_dir, topk_tracking_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(prompt_json),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-weights-json', str(prompt_weights_json),
]

train_cmd = [
    'python', '-u', 'ucf_train_multi_prompt.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--use-pretrained-model', 'false',
    '--train-learnable-prompt', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', str(LAMBDA_CONSISTENCY),
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', '0.20',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('prompt_json:', prompt_json)
print('prompt_weights_json:', prompt_weights_json)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('Manual prompt diagnostics ->', manual_prompt_diag_dir)
print('Top-k tracking ->', topk_tracking_dir)
print('from_scratch: use-pretrained-model=false')
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(direction_name, variant_name, parsed)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

manual_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', output_model,
    '--prompt-json', str(prompt_json),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-weights-json', str(prompt_weights_json),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(manual_prompt_diag_dir),
]
run_command(manual_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'manual_prompt_diag_dir': str(manual_prompt_diag_dir),
    'topk_tracking_dir': str(topk_tracking_dir),
    'prompt_json': str(prompt_json),
    'prompt_weights_json': str(prompt_weights_json),
    'use_pretrained_model': False,
    'trained_from_scratch': True,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('\nAnalyze prompt training outputs:')
print('  mode: weighted')
print('  model:', output_model)
print('  epoch checkpoints:', epoch_dir)
print('  checkpoint diagnostics:', diagnostics_dir)
print('  prompt diagnostics:', manual_prompt_diag_dir)
print('  top-k tracking:', topk_tracking_dir)
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 11. Hướng 2b — Class Prototype `p_c` (gộp Top-K → 1 vector)

**Lý thuyết:**

1. Encode K prompt đã chọn (cùng bank Hướng 2a)
2. Gộp: `p_c = Normalize(Σ_k w_k e_k)` (weighted) hoặc mean nếu không dùng weight
3. Dùng **một** `p_c` / class thay cho class-name / multi-prompt logits
4. Model: `CLIPVADPromptReplacement` (`prompt_replacement`)

Khác Hướng 2a: không còn K kênh logsumexp — chỉ còn **1 prototype embedding**/class.

- Train **from scratch**
- Không load checkpoint
- Learnable prefix/postfix vẫn tồn tại trong kiến trúc nhưng **được đóng băng** trong lần train này

**Kết quả lưu:**
- Evaluate: `code/results/evaluate/ucf_checkpoint_diagnostics_prompt_prototype_*`
- Summary: `code/results/prompt_guided_*.json`


In [ ]:
# === HUONG 2b: Prototype p_c = weighted/mean pool of Top-K prompt embeddings ===
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'

direction_name = 'huong2b_prototype_pc_frozen_prompt'
variant_suffix = 'huong2b_prototype_pc_weighted_frozen_prompt'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
# note: prototype trainer currently has no abn-bg loss; tag kept for naming consistency
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_{reg_tag}'

epoch_dir = f'model/epoch_checkpoints_prompt_prototype_{variant_name}_scratch'
output_model = f'model/model_ucf_prompt_prototype_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_prompt_prototype_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_prompt_prototype_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_prompt_prototype_{variant_name}_scratch'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_proto_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-replacement-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', '-u', 'ucf_train_prompt_replacement.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_proto_args,
    '--use-pretrained-model', 'false',
    '--train-learnable-prompt', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('p_c = Normalize(sum_k w_k * encode(prompt_k))')
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('from_scratch: use-pretrained-model=false')
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'prompt_replacement',
    *common_proto_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={'prototype_pooling': 'weighted_mean_then_l2_normalize'},
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'prompt_replacement',
    *common_proto_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'prototype_pooling': 'weighted_mean_then_l2_normalize',
    'use_pretrained_model': False,
    'trained_from_scratch': True,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 12. Hướng 3 — Learnable Residual Trên Class Prototype

**Công thức:** mỗi step `p_c = encode(bank)` (soft prompt học được), `t_c = Normalize(p_c + Δ_c)`, `Δ_c` init 0.

- Mỗi step: `p_c = encode(prompts)` (learnable soft prompt **vẫn học**), rồi `t_c = Normalize(p_c + Δ_c)`
- Freeze CLIP; train **from scratch** VAD + soft prompt + `Δ_c`
- Loss thêm: `λ_res * mean_c ‖Δ_c‖²` (`LAMBDA_RESIDUAL`, mặc định `1e-3`)
- Không load checkpoint H1/2a/2b
- Không learnable α (v1)

**Kết quả lưu:**
- Evaluate: `code/results/evaluate/ucf_checkpoint_diagnostics_prompt_residual_*`
- Summary: `code/results/prompt_guided_*.json`


In [ ]:
# === HUONG 3: Learnable residual on p_c ===
# t_c = Normalize(encode(prompts) + Delta_c); freeze CLIP; train VAD + soft prompt + Delta from scratch.
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong3_prompt_residual'
variant_suffix = 'huong3_prompt_residual'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

epoch_dir = f'model/epoch_checkpoints_prompt_residual_{variant_name}_scratch'
output_model = f'model/model_ucf_prompt_residual_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_prompt_residual_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_prompt_residual_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_prompt_residual_{variant_name}_scratch'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_proto_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-replacement-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_proto_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('t_c = Normalize(encode(prompts) + Delta_c) | soft prompt trainable')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('from_scratch: use-pretrained-model=false')
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'prompt_residual',
    *common_proto_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'prototype_pooling': 'weighted_mean_then_l2_normalize',
        'residual': 'Normalize(p_c + Delta_c)',
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'prompt_residual',
    *common_proto_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(p_c + Delta_c)',
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': True,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 14. Hướng 2a+3 — Multi-Prompt + Residual Từng Prompt

**Công thức:** mỗi step encode lại rồi cộng residual:

`t_{c,k} = Normalize(encode(prompt_{c,k}) + Δ_{c,k})`

sau đó aggregate có trọng số giống Hướng 2a (weighted logsumexp).

- Freeze CLIP; learnable soft prompt **không** đóng băng
- `Δ` init 0 ⇒ epoch 0 ≈ Hướng 2a (cùng soft prompt)
- Train from scratch VAD + soft prompt + `Δ`
- `λ_res = LAMBDA_RESIDUAL` (mặc định `1e-3`)

**Kết quả lưu:**
- Evaluate: `code/results/evaluate/ucf_checkpoint_diagnostics_multi_prompt_residual_*`
- Analyze: `code/results/analyze/topk_tracking_*`
- Summary: `code/results/prompt_guided_*.json`


In [ ]:
# === HUONG 2a+3: Multi-prompt + per-prompt residual ===
# t_{c,k} = Normalize(encode(prompt_k) + Delta_{c,k}); soft prompt live; weighted aggregate like H2a.
assert USE_PRETRAINED_MODEL is False, 'Prompt finetune must train from scratch.'
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong2a3_multi_prompt_residual'
variant_suffix = 'huong2a3_multi_prompt_residual'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

epoch_dir = f'model/epoch_checkpoints_multi_prompt_residual_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_residual_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_residual_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_residual_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_residual_{variant_name}_scratch'
topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}_scratch'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir, topk_tracking_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_multi_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', str(LAMBDA_CONSISTENCY),
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', '0.20',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('t = Normalize(encode(prompt_k) + Delta_k) | soft prompt trainable; weighted aggregate')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('Top-k tracking ->', topk_tracking_dir)
print('from_scratch: use-pretrained-model=false')
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'topk_tracking_dir': str(topk_tracking_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': True,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 15. Hướng 3-B — Residual Warm-Start từ Hướng 2b (soft prompt + Δ)

**Protocol B:**

1. Load checkpoint **Hướng 2b**
2. `Δ_c` init 0; mỗi step `t_c = Normalize(encode(bank) + Δ_c)` (soft prompt **vẫn học**)
3. Freeze CLIP + VAD; học **soft prompt + `Δ_c`**
4. L2: `λ_res * mean ‖Δ_c‖²`

Cell §12 (H3 from-scratch) giữ nguyên để so sánh.

**Cần trước:** đã chạy §11 (Hướng 2b) với cùng `PROMPT_MODE` / `VARIANT_BASE` / regularize tag.


In [ ]:
# === HUONG 3-B: Warm-start from 2b, train soft prompt + Delta ===
# Load Hướng 2b -> freeze VAD/CLIP; train soft prompt + Delta_c; live encode each step
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong3b_prompt_residual_warmstart'
variant_suffix = 'huong3b_prompt_residual_warmstart_deltaonly'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

# Default base = Hướng 2b output from §11 (same bank / tags). Override if needed.
base_2b_variant = f'{PROMPT_MODE}_{VARIANT_BASE}_huong2b_prototype_pc_weighted_{reg_tag}'
BASE_2B_MODEL = Path(f'model/model_ucf_prompt_prototype_{base_2b_variant}_scratch.pth')
# BASE_2B_MODEL = Path('model/your_custom_2b.pth')  # optional override
assert BASE_2B_MODEL.exists(), (
    f'Hướng 3-B needs Hướng 2b checkpoint. Missing: {BASE_2B_MODEL}\n'
    'Run §11 (Hướng 2b) first, or set BASE_2B_MODEL to an existing path.'
)

epoch_dir = f'model/epoch_checkpoints_prompt_residual_{variant_name}'
output_model = f'model/model_ucf_prompt_residual_{variant_name}.pth'
checkpoint_path = f'model/checkpoint_prompt_residual_{variant_name}.pth'
save_cur_path = f'model/model_cur_prompt_residual_{variant_name}.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_prompt_residual_{variant_name}'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_proto_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-replacement-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_proto_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--train-mode', 'warmstart_delta_only',
    '--base-model-path', str(BASE_2B_MODEL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('base_2b:', BASE_2B_MODEL)
print('t_c = Normalize(encode + Delta_c) | warmstart_delta_only (soft prompt + Delta)')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'prompt_residual',
    *common_proto_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'prototype_pooling': 'weighted_mean_then_l2_normalize',
        'residual': 'Normalize(p_c + Delta_c)',
        'train_mode': 'warmstart_delta_only',
        'base_model': str(BASE_2B_MODEL),
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'prompt_residual',
    *common_proto_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(p_c + Delta_c)',
    'train_mode': 'warmstart_delta_only',
    'base_model': str(BASE_2B_MODEL),
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': False,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 16. Hướng 2a+3-B — Multi-Prompt Residual Warm-Start từ Hướng 2a (soft prompt + Δ)

**Protocol B:**

1. Load checkpoint **Hướng 2a**
2. `Δ_{c,k}` init 0; mỗi step encode lại (soft prompt **vẫn học**)
3. Freeze CLIP + VAD; học **soft prompt + `Δ_k`**
4. Aggregate như 2a (weighted logsumexp) + L2 trên residual

Cell §14 (2a+3 from-scratch) giữ nguyên để so sánh.

**Cần trước:** đã chạy §10 (Hướng 2a) với cùng bank/tags.


In [ ]:
# === HUONG 2a+3-B: Warm-start from 2a, train soft prompt + per-prompt Delta ===
# Load Hướng 2a -> freeze VAD/CLIP; train soft prompt + Delta_k; live encode each step
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong2a3b_multi_prompt_residual_warmstart'
variant_suffix = 'huong2a3b_multi_prompt_residual_warmstart_deltaonly'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

# Default base = Hướng 2a output from §10 (same bank / tags). Override if needed.
base_2a_variant = f'{PROMPT_MODE}_{VARIANT_BASE}_huong2_topk_weighted_{reg_tag}'
BASE_2A_MODEL = Path(f'model/model_ucf_multi_prompt_{base_2a_variant}_scratch.pth')
# BASE_2A_MODEL = Path('model/your_custom_2a.pth')  # optional override
assert BASE_2A_MODEL.exists(), (
    f'Hướng 2a+3-B needs Hướng 2a checkpoint. Missing: {BASE_2A_MODEL}\n'
    'Run §10 (Hướng 2a) first, or set BASE_2A_MODEL to an existing path.'
)

epoch_dir = f'model/epoch_checkpoints_multi_prompt_residual_{variant_name}'
output_model = f'model/model_ucf_multi_prompt_residual_{variant_name}.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_residual_{variant_name}.pth'
save_cur_path = f'model/model_cur_multi_prompt_residual_{variant_name}.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_residual_{variant_name}'
topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir, topk_tracking_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_multi_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--train-mode', 'warmstart_delta_only',
    '--base-model-path', str(BASE_2A_MODEL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', str(LAMBDA_CONSISTENCY),
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', '0.20',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('base_2a:', BASE_2A_MODEL)
print('t = Normalize(encode + Delta_k) | warmstart_delta_only (soft prompt + Delta)')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('Top-k tracking ->', topk_tracking_dir)
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
        'train_mode': 'warmstart_delta_only',
        'base_model': str(BASE_2A_MODEL),
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'topk_tracking_dir': str(topk_tracking_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
    'train_mode': 'warmstart_delta_only',
    'base_model': str(BASE_2A_MODEL),
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': False,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 18. Hướng 3-C — Continue Train từ Checkpoint 2b (VAD + soft prompt + Δ)

**Đúng ý “train tiếp từ 2b”:**

1. Load checkpoint **Hướng 2b**
2. `Δ=0`; mỗi step encode lại (soft prompt học được)
3. **Tiếp tục học VAD + soft prompt + `Δ`** (chỉ freeze CLIP)
4. L2 trên `Δ`

Khác §15 (3-B): 3-B freeze VAD; cell này cập nhật tiếp cả VAD.

**Cần trước:** §11 (2b) cùng bank/tags.


In [ ]:
# === HUONG 3-C: Continue from 2b — train VAD + soft prompt + Delta ===
# Load Hướng 2b -> freeze CLIP only; keep updating VAD + soft prompt + Delta_c
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong3c_prompt_residual_continue'
variant_suffix = 'huong3c_prompt_residual_warmstart_continue'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

base_2b_variant = f'{PROMPT_MODE}_{VARIANT_BASE}_huong2b_prototype_pc_weighted_{reg_tag}'
BASE_2B_MODEL = Path(f'model/model_ucf_prompt_prototype_{base_2b_variant}_scratch.pth')
# BASE_2B_MODEL = Path('model/your_custom_2b.pth')
assert BASE_2B_MODEL.exists(), (
    f'Hướng 3-C needs Hướng 2b checkpoint. Missing: {BASE_2B_MODEL}\n'
    'Run §11 (Hướng 2b) first, or set BASE_2B_MODEL to an existing path.'
)

epoch_dir = f'model/epoch_checkpoints_prompt_residual_{variant_name}'
output_model = f'model/model_ucf_prompt_residual_{variant_name}.pth'
checkpoint_path = f'model/checkpoint_prompt_residual_{variant_name}.pth'
save_cur_path = f'model/model_cur_prompt_residual_{variant_name}.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_prompt_residual_{variant_name}'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_proto_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-replacement-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_proto_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--train-mode', 'warmstart_continue',
    '--base-model-path', str(BASE_2B_MODEL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('base_2b:', BASE_2B_MODEL)
print('t_c = Normalize(encode + Delta_c) | warmstart_continue (VAD + soft prompt + Delta)')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'prompt_residual',
    *common_proto_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'prototype_pooling': 'weighted_mean_then_l2_normalize',
        'residual': 'Normalize(p_c + Delta_c)',
        'train_mode': 'warmstart_continue',
        'base_model': str(BASE_2B_MODEL),
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'prompt_residual',
    *common_proto_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(p_c + Delta_c)',
    'train_mode': 'warmstart_continue',
    'base_model': str(BASE_2B_MODEL),
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': False,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 19. Hướng 2a+3-C — Continue Train từ Checkpoint 2a (VAD + soft prompt + Δ)

**Đúng ý “train tiếp từ 2a”:**

1. Load checkpoint **Hướng 2a**
2. `Δ=0`; mỗi step encode lại (soft prompt học được)
3. **Tiếp tục học VAD + soft prompt + `Δ_k`** (chỉ freeze CLIP)
4. Aggregate như 2a + L2 trên residual

Khác §16 (2a+3-B): B freeze VAD; cell này cập nhật tiếp cả VAD.

**Cần trước:** §10 (2a) cùng bank/tags.


In [ ]:
# === HUONG 2a+3-C: Continue from 2a — train VAD + soft prompt + per-prompt Delta ===
# Load Hướng 2a -> freeze CLIP only; keep updating VAD + soft prompt + Delta_k
assert 'EVALUATE_RESULTS_DIR' in globals() and 'ANALYZE_RESULTS_DIR' in globals()
assert 'LAMBDA_RESIDUAL' in globals(), 'Re-run config cell to define LAMBDA_RESIDUAL'

direction_name = 'huong2a3c_multi_prompt_residual_continue'
variant_suffix = 'huong2a3c_multi_prompt_residual_warmstart_continue'
reg_tag = 'abn_bg' if LAMBDA_ABNORMAL_BG > 0 else 'no_regularize'
variant_name = f'{PROMPT_MODE}_{VARIANT_BASE}_{variant_suffix}_lres{LAMBDA_RESIDUAL:g}_{reg_tag}'

base_2a_variant = f'{PROMPT_MODE}_{VARIANT_BASE}_huong2_topk_weighted_{reg_tag}'
BASE_2A_MODEL = Path(f'model/model_ucf_multi_prompt_{base_2a_variant}_scratch.pth')
# BASE_2A_MODEL = Path('model/your_custom_2a.pth')
assert BASE_2A_MODEL.exists(), (
    f'Hướng 2a+3-C needs Hướng 2a checkpoint. Missing: {BASE_2A_MODEL}\n'
    'Run §10 (Hướng 2a) first, or set BASE_2A_MODEL to an existing path.'
)

epoch_dir = f'model/epoch_checkpoints_multi_prompt_residual_{variant_name}'
output_model = f'model/model_ucf_multi_prompt_residual_{variant_name}.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_residual_{variant_name}.pth'
save_cur_path = f'model/model_cur_multi_prompt_residual_{variant_name}.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_residual_{variant_name}'
topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}'

for result_dir in [EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR, diagnostics_dir, topk_tracking_dir]:
    Path(result_dir).mkdir(parents=True, exist_ok=True)

for stale in [Path(checkpoint_path), Path(save_cur_path)]:
    if stale.exists():
        stale.unlink()
        print('Removed stale checkpoint:', stale)

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(PROMPT_BANK_JSON),
    '--prompt-mode', PROMPT_MODE,
    '--prompt-weights-json', str(PROMPT_BANK_JSON),
]

train_cmd = [
    'python', 'ucf_train_multi_prompt_residual.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--lambda-residual', str(LAMBDA_RESIDUAL),
    '--train-mode', 'warmstart_continue',
    '--base-model-path', str(BASE_2A_MODEL),
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', str(LAMBDA_CONSISTENCY),
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', '0.20',
    '--lr', str(LR),
    '--max-epoch', str(MAX_EPOCH),
    '--scheduler-milestones', *SCHEDULER_MILESTONES,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]

print('\n' + '=' * 100)
print(direction_name)
print('variant:', variant_name)
print('base_2a:', BASE_2A_MODEL)
print('t = Normalize(encode + Delta_k) | warmstart_continue (VAD + soft prompt + Delta)')
print('lambda_residual:', LAMBDA_RESIDUAL)
print('Evaluate dir:', EVALUATE_RESULTS_DIR)
print('Analyze dir:', ANALYZE_RESULTS_DIR)
print('Checkpoint diagnostics ->', diagnostics_dir)
print('Top-k tracking ->', topk_tracking_dir)
run_command(train_cmd, capture=False)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
eval_stdout = run_command(eval_cmd, capture=True)
parsed = parse_evaluate_stdout(eval_stdout)
result_record = print_finetune_result(
    direction_name,
    variant_name,
    parsed,
    extra={
        'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
        'train_mode': 'warmstart_continue',
        'base_model': str(BASE_2A_MODEL),
        'lambda_residual': LAMBDA_RESIDUAL,
    },
)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt_residual',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
run_command(analyze_cmd, capture=False)

info = {
    'direction': direction_name,
    'variant_name': variant_name,
    'model': output_model,
    'epoch_dir': epoch_dir,
    'diagnostics_dir': str(diagnostics_dir),
    'topk_tracking_dir': str(topk_tracking_dir),
    'prompt_json': str(PROMPT_BANK_JSON),
    'prompt_weights_json': str(PROMPT_BANK_JSON),
    'residual': 'Normalize(e_k + Delta_k) then weighted multi-prompt',
    'train_mode': 'warmstart_continue',
    'base_model': str(BASE_2A_MODEL),
    'lambda_residual': LAMBDA_RESIDUAL,
    'use_pretrained_model': False,
    'trained_from_scratch': False,
    'metrics': result_record,
}
RUN_OUTPUTS[direction_name] = info
summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
summary_path.write_text(json.dumps(RUN_OUTPUTS, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
metrics_path.write_text(json.dumps(FINETUNE_RESULTS, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved summary:', summary_path)
print('Saved metrics:', metrics_path)
print_all_finetune_results()


## 20. Tóm Tắt Các Hướng Đã Chạy

So kết quả với baseline tự train (~87.19) và auc8715 (~87.15).

| Nhóm | Ý nghĩa |
|---|---|
| 3 / 2a+3 | From scratch (VAD + soft prompt + Δ) |
| 3-B / 2a+3-B | Warm-start, freeze VAD; học soft prompt + Δ |
| 3-C / 2a+3-C | Warm-start từ 2b/2a; tiếp tục VAD + soft prompt + Δ |


In [ ]:
print_all_finetune_results()

print('Completed / recorded runs:')
for name, info in RUN_OUTPUTS.items():
    print('\n' + name)
    metrics = info.get('metrics') or {}
    model_m = metrics.get('model') or {}
    if model_m:
        print(
            f"  metrics: cls_auc={model_m.get('classifier_auc', float('nan')):.2f} "
            f"cls_ap={model_m.get('classifier_ap', float('nan')):.2f} "
            f"ano_auc={model_m.get('ano_auc', float('nan')):.2f} "
            f"avg_mAP={model_m.get('avg_mAP', float('nan')):.2f}"
        )
    for key, value in info.items():
        if key == 'metrics':
            continue
        print(f'  {key}: {value}')

summary_path = RESULTS_ROOT / 'prompt_guided_run_summary.json'
metrics_path = RESULTS_ROOT / 'prompt_guided_finetune_results.json'
if summary_path.exists():
    print('\nSummary file:', summary_path)
if metrics_path.exists():
    print('Metrics file:', metrics_path)

print('\nNext: analyze_vadclip_class_prompt_alignment.ipynb')
print('Use the prompt JSON + checkpoint from the direction you want to inspect.')
